# Notebook 2 of 5 · Baseline ML — TF-IDF + Logistic Regression
### The classical-ML *control floor*

> **Why a classical baseline matters for LLM governance.** Before claiming an LLM is a
> *good* safety classifier, you need a credible *floor* to measure it against. A cheap,
> fast, fully-interpretable TF-IDF + linear model is that floor — the **champion** the
> Claude **challenger** (NB3) must beat to justify its cost, latency, and opacity. It is
> also the on-prem fallback control if the API is unavailable. Logic: [ml_baseline.py](../src/ml_baseline.py).

In [ ]:
# --- repo bootstrap: make `from src import ...` work from notebooks/ ---
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.titleweight": "bold", "font.size": 10})

import src
from src import data_loader as dl
from src.data_loader import LABELS, ANTHROPIC_POLICY_MAP, SEVERITY_ORDER
print("src loaded. LABELS:", LABELS)

In [ ]:
from src import ml_baseline as mlb
from src import safety_metrics as sm
df = dl.load_jigsaw()
print(f"Corpus: {len(df):,} comments")

## 1 · Train the baseline

> `class_weight='balanced'` is a **governance choice**, not a default: at ~90% clean, an
> unweighted model optimises for the majority (clean) class and under-detects the rare,
> high-severity harms we care most about.

In [ ]:
pipe, splits = mlb.train_baseline(df)          # TF-IDF(1-2gram) + OvR LogReg, balanced
proba, pred = mlb.predict(pipe, splits["X_test"])
model_path = mlb.save_model(pipe)
print("Trained & saved:", model_path)
print("Test set:", len(splits['X_test']), "comments")

## 2 · Severity-tiered scorecard

> We report **per-category precision/recall/F1 + AUCs**, annotated with the Anthropic
> severity tier — a governance scorecard, not a bare ML report. Accuracy is deliberately
> omitted as a headline (it would read ~98% and mean nothing).

In [ ]:
scorecard = sm.per_category_metrics(splits["Y_test"].values, pred.values, proba.values)
scorecard.to_csv(os.path.join(dl.outputs_dir(), "ml_baseline_scorecard.csv"), index=False)
scorecard.round(3)

In [ ]:
# Recall by category, coloured by severity — the binding-constraint view
sc = scorecard.copy()
colmap = {"Critical":"#C44E52","High":"#DD8452","Medium":"#4C72B0"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].bar(sc["category"], sc["recall"], color=[colmap[t] for t in sc["severity_tier"]])
axes[0].set_title("Recall by category (colour = severity)"); axes[0].set_ylabel("recall")
axes[0].set_ylim(0, 1); axes[0].tick_params(axis="x", rotation=30)
axes[1].bar(sc["category"], sc["f1"], color=[colmap[t] for t in sc["severity_tier"]])
axes[1].set_title("F1 by category"); axes[1].set_ylim(0, 1); axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(os.path.join(dl.results_dir(), "ml_baseline_performance.png"),
                                bbox_inches="tight"); plt.show()
print("macro/micro:\n", sm.macro_micro_summary(splits["Y_test"].values, pred.values).round(3).to_string())

**Read-out.** ROC-AUCs are high across the board, but **recall on the Critical-severity
rare classes (`threat`, `identity_hate`) is the binding constraint** — precisely the harms
the data is thinnest on. This is the governance story: a model can look excellent on AUC
and still under-protect on the harms that matter most. NB5 attacks this with severity-tiered
thresholds.

## 3 · Confusion matrices (per category)

In [ ]:
cms = sm.confusion_per_category(splits["Y_test"].values, pred.values)
fig, axes = plt.subplots(2, 3, figsize=(11, 6.5))
for ax, lab in zip(axes.ravel(), LABELS):
    m = cms[lab]
    im = ax.imshow(m, cmap="Blues")
    ax.set_title(f"{lab}\n[{ANTHROPIC_POLICY_MAP[lab]['severity_tier']}]", fontsize=9)
    ax.set_xticks([0,1]); ax.set_xticklabels(["pred 0","pred 1"], fontsize=8)
    ax.set_yticks([0,1]); ax.set_yticklabels(["true 0","true 1"], fontsize=8)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{m[i,j]:,}", ha="center", va="center",
                    color="white" if m[i,j] > m.max()/2 else "black", fontsize=8)
    ax.grid(False)
plt.tight_layout(); plt.savefig(os.path.join(dl.results_dir(), "category_confusion_matrix.png"),
                                bbox_inches="tight"); plt.show()

## 4 · Interpretability — *why* does it flag?

> A linear model's coefficients are an audit surface: a reviewer can read the most
> influential tokens per category and sanity-check that the control keys on harm, not on a
> spurious correlate. This transparency is an EU AI Act Art. 13 asset the LLM challenger
> lacks.

In [ ]:
# Slur tokens are redacted on display — the safety system should not reproduce the
# content it detects, even in its own interpretability output.
for lab in ["threat", "identity_hate", "insult"]:
    toks = [dl.redact(t) for t in mlb.top_features(pipe, lab, 10)["token"]]
    print(f"{lab:14s}: {toks}")

## 5 · Thresholds as a governance lever

> A single 0.5 threshold treats a missed `threat` like a missed `obscene`. Below we apply
> **severity-tiered thresholds** (Critical 0.30 / High 0.40 / Medium 0.50) and measure the
> recall lift on the Critical harms — the KB-Life-KRI logic, previewed here and formalised
> in NB5.

In [ ]:
pred_tiered = sm.apply_tiered_thresholds(proba.values)
sc_flat = sm.per_category_metrics(splits["Y_test"].values, pred.values).set_index("category")
sc_tier = sm.per_category_metrics(splits["Y_test"].values, pred_tiered).set_index("category")
cmp = pd.DataFrame({
    "severity": [ANTHROPIC_POLICY_MAP[l]["severity_tier"] for l in LABELS],
    "recall_flat_0.5": sc_flat.loc[LABELS, "recall"].values,
    "recall_tiered":   sc_tier.loc[LABELS, "recall"].values,
}, index=LABELS)
cmp["recall_lift"] = (cmp["recall_tiered"] - cmp["recall_flat_0.5"]).round(3)
cmp.round(3)

**Read-out.** Lowering the threshold on Critical categories lifts their recall (catching
more true harms) at the cost of some precision — the right trade when a missed threat is a
real-world safety event. This is *policy*, expressed in code.

## 6 · Summary

- Baseline **champion floor** established and saved (`models/tfidf_logreg_baseline.joblib`);
  scorecard saved to `outputs/`.
- High AUC, but **Critical-harm recall is the binding constraint** — the gap the Claude
  challenger (NB3) is introduced to close.
- The model is **interpretable** (token weights) and **threshold-governable** (tiered cuts).

*Next → **Notebook 3: Claude API classification** — the policy-native challenger.*